# DS4200 – Visualizations V3 & V4
**V3:** Boston CO concentration over time (2025)  
**V4:** Median CO concentration by U.S. region (2025)

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

---
## V3 – Boston CO Concentration Over Time (2025)

Scatter plot of daily CO readings across Boston metro monitors, colored by month.
Helps reveal seasonal trends and day-to-day variability in carbon monoxide levels.

In [ ]:
# --- Load & filter CO data for Boston metro ---
co = pd.read_csv("daily_42101_2025.csv", low_memory=False)
co.columns = co.columns.str.strip()

bos_co = co[
    co["CBSA Name"] == "Boston-Cambridge-Newton, MA-NH"
].copy()

bos_co["Date Local"] = pd.to_datetime(bos_co["Date Local"])

# Daily mean across all Boston monitors
daily_co = (
    bos_co.groupby("Date Local", as_index=False)
    .agg(co_ppm=("Arithmetic Mean", "mean"), aqi=("AQI", "mean"))
    .sort_values("Date Local")
)

daily_co["month_name"] = daily_co["Date Local"].dt.strftime("%B")

daily_co.head()

In [ ]:
# --- Build V3 chart ---

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

# LOESS trend line
trend = alt.Chart(daily_co).transform_loess(
    "Date Local", "co_ppm", bandwidth=0.3
).mark_line(
    color="#555", strokeDash=[5, 3], strokeWidth=1.5, opacity=0.6
).encode(
    x="Date Local:T",
    y="co_ppm:Q"
)

# Scatter points colored by month
points = alt.Chart(daily_co).mark_circle(size=65, opacity=0.85).encode(
    x=alt.X(
        "Date Local:T",
        title="Date",
        axis=alt.Axis(format="%b", tickCount="month", labelAngle=0)
    ),
    y=alt.Y(
        "co_ppm:Q",
        title="Daily Avg CO (ppm)",
        scale=alt.Scale(zero=False)
    ),
    color=alt.Color(
        "month_name:N",
        title="Month",
        sort=month_order,
        scale=alt.Scale(scheme="turbo"),
        legend=alt.Legend(orient="right")
    ),
    tooltip=[
        alt.Tooltip("Date Local:T", title="Date",        format="%b %d, %Y"),
        alt.Tooltip("co_ppm:Q",     title="CO (ppm)",    format=".3f"),
        alt.Tooltip("aqi:Q",        title="Avg AQI",     format=".1f"),
        alt.Tooltip("month_name:N", title="Month")
    ]
)

v3 = alt.layer(trend, points).properties(
    title=alt.TitleParams(
        text="Boston Area CO Concentration Over Time (2025)",
        fontSize=16,
        subtitleFontSize=12,
        anchor="start"
    ),
    width=900,
    height=420
).interactive()

v3

In [ ]:
v3.save("boston_co_over_time.html")

---
## V4 – Median CO Concentration by U.S. Region (2025)

Bar chart showing the median of daily CO readings aggregated across all monitors in each geographic region.
Sorted descending so the highest-CO regions stand out immediately.

In [ ]:
# --- Load CO data & assign regions ---
co = pd.read_csv("daily_42101_2025.csv", low_memory=False)
co.columns = co.columns.str.strip()

region_map = {
    # Northeast
    "Maine": "Northeast", "New Hampshire": "Northeast", "Vermont": "Northeast",
    "Massachusetts": "Northeast", "Rhode Island": "Northeast", "Connecticut": "Northeast",
    "New York": "Northeast", "New Jersey": "Northeast", "Pennsylvania": "Northeast",
    # Mid-Atlantic
    "Maryland": "Mid-Atlantic", "Delaware": "Mid-Atlantic",
    "District Of Columbia": "Mid-Atlantic", "Virginia": "Mid-Atlantic",
    "West Virginia": "Mid-Atlantic",
    # Southeast
    "North Carolina": "Southeast", "South Carolina": "Southeast",
    "Georgia": "Southeast", "Florida": "Southeast", "Alabama": "Southeast",
    "Mississippi": "Southeast", "Tennessee": "Southeast", "Kentucky": "Southeast",
    # South Central
    "Texas": "South Central", "Oklahoma": "South Central",
    "Arkansas": "South Central", "Louisiana": "South Central",
    # Midwest
    "Ohio": "Midwest", "Indiana": "Midwest", "Illinois": "Midwest",
    "Michigan": "Midwest", "Wisconsin": "Midwest", "Minnesota": "Midwest",
    "Iowa": "Midwest", "Missouri": "Midwest", "North Dakota": "Midwest",
    "South Dakota": "Midwest", "Nebraska": "Midwest", "Kansas": "Midwest",
    # Mountain West
    "Montana": "Mountain West", "Idaho": "Mountain West", "Wyoming": "Mountain West",
    "Colorado": "Mountain West", "New Mexico": "Mountain West",
    "Arizona": "Mountain West", "Utah": "Mountain West", "Nevada": "Mountain West",
    # Pacific Northwest
    "Washington": "Pacific Northwest", "Oregon": "Pacific Northwest",
    # California on its own (large + distinct air quality issues)
    "California": "California",
    # Alaska / Hawaii
    "Alaska": "Alaska / Hawaii", "Hawaii": "Alaska / Hawaii",
    # Territories
    "Puerto Rico": "Territories",
}

co["Region"] = co["State Name"].map(region_map)

# Median daily CO per region
regional = (
    co.dropna(subset=["Region", "Arithmetic Mean"])
    .groupby("Region", as_index=False)["Arithmetic Mean"]
    .median()
    .rename(columns={"Arithmetic Mean": "Median CO (ppm)"})
    .sort_values("Median CO (ppm)", ascending=False)
)

regional

In [ ]:
# --- Build V4 chart ---

# Sort order for y-axis (highest → lowest)
region_order = regional["Region"].tolist()

bars = alt.Chart(regional).mark_bar(cornerRadiusEnd=4).encode(
    y=alt.Y(
        "Region:N",
        sort=region_order,
        title=None,
        axis=alt.Axis(labelFontSize=12)
    ),
    x=alt.X(
        "Median CO (ppm):Q",
        title="Median CO Concentration (ppm)",
        scale=alt.Scale(zero=True)
    ),
    color=alt.Color(
        "Median CO (ppm):Q",
        scale=alt.Scale(scheme="orangered"),
        legend=None
    ),
    tooltip=[
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Median CO (ppm):Q", title="Median CO (ppm)", format=".4f")
    ]
)

# Value labels on bars
labels = bars.mark_text(align="left", dx=4, fontSize=11, color="#444").encode(
    text=alt.Text("Median CO (ppm):Q", format=".3f")
)

v4 = alt.layer(bars, labels).properties(
    title=alt.TitleParams(
        text="Median Daily CO Concentration by U.S. Region (2025)",
        fontSize=16,
        subtitleFontSize=12,
        anchor="start"
    ),
    width=600,
    height=360
).configure_view(
    stroke=None
).configure_axis(
    grid=True,
    gridColor="#eeeeee"
)

v4

In [ ]:
v4.save("median_co_by_region.html")